# Notebook 1 — Gamry import and experiment validation

**Purpose.** Read the original Gamry `.DTA` files, reconstruct the polarization-curve
experiment, check that the dataset is complete and internally consistent, and export
clean machine-readable tables for Notebook 2.

This notebook intentionally performs **no electrochemical analysis**: it does not average
CP potentials, select CV cycles, fit EIS, apply iR correction, calculate Tafel slopes, or
create report figures.

Expected experiment:

1. OCP
2. Activation CP (60 mA, 8 h)
3. Three initial EIS replicates
4. Initial CV
5. Preconditioning CP (1 mA cm⁻²)
6. Repeated CP → EIS → CV polarization steps
7. Three final EIS replicates
8. Final CV

## 1. Configuration

Put all raw `.DTA` files for **one experiment** in a single folder. Change only
`RAW_DATA_DIR` below. On Windows, use a raw string such as
`Path(r"C:\Users\Daniel\Documents\My experiment")`.

The default expects a folder named `raw_data` beside this notebook. Generated files go
into `processed_data`; the raw files are never modified.

In [ ]:
from pathlib import Path
import csv
import json
import re
import sys
import warnings
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

RAW_DATA_DIR = Path.cwd() / "raw_data"
OUTPUT_DIR = Path.cwd() / "processed_data"

EXPERIMENT = {
    "experiment_name": "Ni mesh in 7 M Fe-unpurified KOH",
    "working_electrode": "Ni mesh",
    "counter_electrode": "Ni mesh",
    "reference_electrode": "Hg/HgO",
    "reference_filling_solution": "7 M KOH",
    "electrolyte": "7 M KOH",
    "koh_condition": "Fe-unpurified",
    "temperature_C": 25.0,
    "flow_rate_mL_min": 100.0,
    "geometric_area_cm2": 4.0,
    "activation_current_mA": 60.0,
    "activation_time_h": 8.0,
    "polarization_cp_duration_min": 10.0,
    "eis_frequency_high_Hz": 100_000.0,
    "eis_frequency_low_Hz": 0.8,
}

EXPECTED_FIXED_COUNTS = {
    "ocp": 1,
    "activation_cp": 1,
    "initial_eis": 3,
    "initial_cv": 1,
    "preconditioning_cp": 1,
    "final_eis": 3,
    "final_cv": 1,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw-data folder: {RAW_DATA_DIR.resolve()}")
print(f"Output folder:   {OUTPUT_DIR.resolve()}")

## 2. Gamry text parser

Gamry `.DTA` files are tab-delimited text files containing a header followed by one or
more named data tables (commonly `CURVE`, `ZCURVE`, or `OCVCURVE`). The functions below:

- try the common Gamry encodings without altering the source;
- retain header metadata;
- detect every numeric table;
- preserve the original column names and units;
- coerce numeric values safely and report malformed rows.

In [ ]:
TABLE_MARKER_RE = re.compile(
    r"^\s*(?P<name>[A-Za-z][A-Za-z0-9_ -]*(?:CURVE|TABLE))\s*(?:\t|$)",
    re.IGNORECASE,
)


def read_gamry_text(path):
    """Read a Gamry text file using common encodings and return text + encoding."""
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Could not decode {path.name}")


def split_fields(line):
    """Split a Gamry line, preferring tabs and retaining multiword labels."""
    line = line.rstrip("\r\n")
    return [field.strip() for field in line.split("\t")] if "\t" in line else line.split()


def is_number(value):
    try:
        float(str(value).replace(",", ""))
        return True
    except (TypeError, ValueError):
        return False


def unique_columns(columns):
    seen, result = Counter(), []
    for index, column in enumerate(columns):
        name = str(column).strip() or f"column_{index}"
        seen[name] += 1
        result.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return result


def extract_header(lines, first_table_line):
    """Store pre-table metadata without assuming a fixed Gamry header layout."""
    metadata = {}
    unparsed = []
    for line in lines[:first_table_line]:
        fields = split_fields(line)
        nonempty = [f for f in fields if f]
        if len(nonempty) >= 2:
            key = nonempty[0].rstrip(":")
            value = " | ".join(nonempty[1:])
            if key in metadata:
                metadata[key] = (
                    metadata[key] + [value]
                    if isinstance(metadata[key], list)
                    else [metadata[key], value]
                )
            else:
                metadata[key] = value
        elif nonempty:
            unparsed.append(nonempty[0])
    if unparsed:
        metadata["_unparsed_header_lines"] = unparsed
    return metadata


def parse_table(lines, marker_index, end_index):
    marker_fields = split_fields(lines[marker_index])
    marker = marker_fields[0].strip() if marker_fields else "TABLE"

    header_index = None
    for i in range(marker_index + 1, min(marker_index + 6, end_index)):
        fields = split_fields(lines[i])
        if len(fields) >= 2 and sum(is_number(v) for v in fields) < len(fields) / 2:
            header_index = i
            break
    if header_index is None:
        return marker, pd.DataFrame(), {}, ["Column header row not found"]

    columns = unique_columns(split_fields(lines[header_index]))
    units, data_start = {}, header_index + 1
    if data_start < end_index:
        possible_units = split_fields(lines[data_start])
        if len(possible_units) == len(columns) and not any(is_number(v) for v in possible_units):
            units = dict(zip(columns, possible_units))
            data_start += 1

    rows, malformed = [], []
    for line_number in range(data_start, end_index):
        fields = split_fields(lines[line_number])
        if not fields or not any(is_number(v) for v in fields):
            continue
        if len(fields) != len(columns):
            malformed.append(line_number + 1)
            fields = (fields + [None] * len(columns))[:len(columns)]
        rows.append(fields)

    frame = pd.DataFrame(rows, columns=columns)
    for column in frame.columns:
        converted = pd.to_numeric(
            frame[column].astype(str).str.replace(",", "", regex=False),
            errors="coerce",
        )
        if converted.notna().sum() >= max(1, int(0.8 * frame[column].notna().sum())):
            frame[column] = converted
    notes = [f"Malformed/padded rows at source lines: {malformed}"] if malformed else []
    return marker, frame, units, notes


def parse_gamry_dta(path):
    text, encoding = read_gamry_text(path)
    lines = text.splitlines()
    markers = [i for i, line in enumerate(lines) if TABLE_MARKER_RE.match(line)]
    if not markers:
        # Conservative fallback for uncommon files whose marker omits CURVE/TABLE.
        markers = [
            i for i, line in enumerate(lines)
            if split_fields(line) and split_fields(line)[0].upper() in
            {"CURVE", "ZCURVE", "OCVCURVE", "READZ", "CHRONO"}
        ]
    first_table = markers[0] if markers else len(lines)
    parsed = {
        "path": path,
        "encoding": encoding,
        "header": extract_header(lines, first_table),
        "tables": [],
        "warnings": [],
        "line_count": len(lines),
    }
    if not markers:
        parsed["warnings"].append("No recognized numeric table marker found")
        return parsed
    boundaries = markers[1:] + [len(lines)]
    for marker_index, end_index in zip(markers, boundaries):
        name, frame, units, notes = parse_table(lines, marker_index, end_index)
        parsed["tables"].append({"name": name, "data": frame, "units": units})
        parsed["warnings"].extend(notes)
    return parsed

## 3. File classification and sequence reconstruction

Classification uses filenames, not acquisition order guesses. It recognizes the established
names (`OCP_Pre`, `CPAct`, `EIS_initial_#1`, `CV-initial`, `CP_1mA-Pre`, and corresponding
step/final files). Unrecognized files remain in the manifest for manual review.

In [ ]:
def natural_key(text):
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", str(text))]


def classify_file(path):
    stem = path.stem
    normalized = re.sub(r"[\s_-]+", " ", stem.lower()).strip()

    if "ocp" in normalized:
        stage = "ocp"
    elif re.search(r"\b(cpact|cp act|activation)\b", normalized):
        stage = "activation_cp"
    elif "eis" in normalized and re.search(r"\b(initial|pre)\b", normalized):
        stage = "initial_eis"
    elif "cv" in normalized and re.search(r"\b(initial|pre)\b", normalized):
        stage = "initial_cv"
    elif "cp" in normalized and (
        re.search(r"\b1\s*(ma|ma cm)\b", normalized) and "pre" in normalized
    ):
        stage = "preconditioning_cp"
    elif "eis" in normalized and re.search(r"\b(final|post)\b", normalized):
        stage = "final_eis"
    elif "cv" in normalized and re.search(r"\b(final|post)\b", normalized):
        stage = "final_cv"
    elif "cp" in normalized:
        stage = "step_cp"
    elif "eis" in normalized:
        stage = "step_eis"
    elif "cv" in normalized:
        stage = "step_cv"
    else:
        stage = "unclassified"

    replicate_match = re.search(r"(?:#|rep(?:licate)?\s*)?(\d+)\s*$", normalized)
    current_match = re.search(
        r"(?<![a-z])(?P<value>\d+(?:\.\d+)?)\s*"
        r"(?P<unit>ma(?:\s*(?:cm-?2|cm\^-?2|cm²))?|a(?:\s*(?:cm-?2|cm\^-?2|cm²))?)",
        normalized,
    )
    current_value = float(current_match.group("value")) if current_match else np.nan
    current_unit = current_match.group("unit").replace(" ", "") if current_match else None

    return {
        "stage": stage,
        "replicate": int(replicate_match.group(1)) if replicate_match else np.nan,
        "filename_current_value": current_value,
        "filename_current_unit": current_unit,
    }


STAGE_ORDER = {
    "ocp": 0,
    "activation_cp": 1,
    "initial_eis": 2,
    "initial_cv": 3,
    "preconditioning_cp": 4,
    "step_cp": 5,
    "step_eis": 6,
    "step_cv": 7,
    "final_eis": 8,
    "final_cv": 9,
    "unclassified": 99,
}


def sequence_sort_key(record):
    stage = record["stage"]
    current = record["filename_current_value"]
    replicate = record["replicate"]
    # Within polarization steps, current is the primary grouping variable.
    if stage.startswith("step_"):
        technique_order = {"step_cp": 0, "step_eis": 1, "step_cv": 2}[stage]
        return (5, np.inf if pd.isna(current) else current, technique_order,
                np.inf if pd.isna(replicate) else replicate, natural_key(record["filename"]))
    return (STAGE_ORDER[stage], 0, 0,
            np.inf if pd.isna(replicate) else replicate, natural_key(record["filename"]))

## 4. Discover and parse the raw files

In [ ]:
dta_files = (
    sorted(RAW_DATA_DIR.rglob("*.DTA"), key=lambda p: natural_key(p.name))
    + sorted(RAW_DATA_DIR.rglob("*.dta"), key=lambda p: natural_key(p.name))
    if RAW_DATA_DIR.exists()
    else []
)
# Remove duplicates on case-insensitive filesystems.
dta_files = list(dict.fromkeys(path.resolve() for path in dta_files))

if not dta_files:
    warnings.warn(
        f"No .DTA files found in {RAW_DATA_DIR.resolve()}. "
        "Set RAW_DATA_DIR in Section 1, then run all cells again."
    )
else:
    print(f"Found {len(dta_files)} raw Gamry files.")

parsed_files, manifest_rows = {}, []
for path in dta_files:
    parsed = parse_gamry_dta(path)
    parsed_files[path.name] = parsed
    classification = classify_file(path)
    nonempty_tables = [t for t in parsed["tables"] if not t["data"].empty]
    row = {
        "filename": path.name,
        "relative_path": str(path.relative_to(RAW_DATA_DIR.resolve())),
        **classification,
        "encoding": parsed["encoding"],
        "source_lines": parsed["line_count"],
        "table_count": len(parsed["tables"]),
        "nonempty_table_count": len(nonempty_tables),
        "table_names": " | ".join(t["name"] for t in parsed["tables"]),
        "data_rows": sum(len(t["data"]) for t in parsed["tables"]),
        "parser_warning_count": len(parsed["warnings"]),
        "parser_warnings": " | ".join(parsed["warnings"]),
    }
    manifest_rows.append(row)

manifest = pd.DataFrame(manifest_rows)
if not manifest.empty:
    manifest = manifest.sort_values(
        by=list(manifest.columns),
        key=lambda col: col.map(natural_key) if col.name == "filename" else col,
    ).reset_index(drop=True)
    ordered_names = sorted(manifest.to_dict("records"), key=sequence_sort_key)
    sequence_lookup = {row["filename"]: i + 1 for i, row in enumerate(ordered_names)}
    manifest.insert(0, "sequence_index", manifest["filename"].map(sequence_lookup))
    manifest = manifest.sort_values("sequence_index").reset_index(drop=True)

display(manifest)

## 5. Completeness, sequence, and metadata checks

`PASS` means the fixed expected file count was found. `REVIEW` is deliberately used for
conditions that require human judgment, such as unclassified names, missing current labels,
or an unusually short OCP caused by Gamry's stability criterion.

In [ ]:
validation_rows = []


def add_check(check, status, observed, expected, detail=""):
    validation_rows.append({
        "check": check,
        "status": status,
        "observed": observed,
        "expected": expected,
        "detail": detail,
    })


if manifest.empty:
    add_check("Raw files discovered", "FAIL", 0, "> 0",
              "Set RAW_DATA_DIR and rerun the notebook.")
else:
    add_check("Raw files discovered", "PASS", len(manifest), "> 0")
    counts = manifest["stage"].value_counts().to_dict()
    for stage, expected in EXPECTED_FIXED_COUNTS.items():
        observed = counts.get(stage, 0)
        add_check(
            f"Count: {stage}",
            "PASS" if observed == expected else "FAIL",
            observed,
            expected,
        )

    unclassified = manifest.loc[
        manifest["stage"].eq("unclassified"), "filename"
    ].tolist()
    add_check(
        "All filenames classified",
        "PASS" if not unclassified else "REVIEW",
        len(unclassified),
        0,
        ", ".join(unclassified),
    )

    empty_tables = manifest.loc[
        manifest["nonempty_table_count"].eq(0), "filename"
    ].tolist()
    add_check(
        "Every file has a numeric table",
        "PASS" if not empty_tables else "FAIL",
        len(empty_tables),
        0,
        ", ".join(empty_tables),
    )

    parser_warnings = manifest.loc[
        manifest["parser_warning_count"].gt(0), "filename"
    ].tolist()
    add_check(
        "Parser warnings",
        "PASS" if not parser_warnings else "REVIEW",
        len(parser_warnings),
        0,
        ", ".join(parser_warnings),
    )

    for step_stage in ("step_cp", "step_eis", "step_cv"):
        subset = manifest.loc[manifest["stage"].eq(step_stage)]
        missing = subset.loc[subset["filename_current_value"].isna(), "filename"].tolist()
        add_check(
            f"Current labels present: {step_stage}",
            "PASS" if not missing else "REVIEW",
            len(subset) - len(missing),
            len(subset),
            ", ".join(missing),
        )

    # OCP duration check uses the maximum time-like column from its first numeric table.
    ocp_names = manifest.loc[manifest["stage"].eq("ocp"), "filename"].tolist()
    if ocp_names:
        ocp_tables = parsed_files[ocp_names[0]]["tables"]
        duration_s = np.nan
        for table in ocp_tables:
            time_columns = [
                c for c in table["data"].columns
                if re.search(r"(^|[^a-z])(t|time)([^a-z]|$)", str(c), re.I)
            ]
            if time_columns:
                duration_s = pd.to_numeric(
                    table["data"][time_columns[0]], errors="coerce"
                ).max()
                break
        status = "PASS" if pd.notna(duration_s) and duration_s >= 590 else "REVIEW"
        detail = (
            "A short OCP can be valid when Gamry's 1 mV/s stability criterion "
            "advances the sequence before the 600 s maximum."
        )
        add_check("OCP recorded duration", status, duration_s, "up to 600 s", detail)

validation = pd.DataFrame(validation_rows)
display(validation.style.applymap(
    lambda value: (
        "background-color: #d9ead3" if value == "PASS"
        else "background-color: #fff2cc" if value == "REVIEW"
        else "background-color: #f4cccc" if value == "FAIL"
        else ""
    )
))

## 6. Header metadata review

This table exposes the original Gamry header fields side-by-side. It is intentionally
lossless and wide: use it to spot changes in instrument settings, sample labels, acquisition
times, or requested current/frequency settings before continuing to Notebook 2.

In [ ]:
header_rows = []
for filename, parsed in parsed_files.items():
    row = {"filename": filename, **parsed["header"]}
    header_rows.append(row)

headers = pd.DataFrame(header_rows)
if not headers.empty and not manifest.empty:
    headers.insert(0, "sequence_index", headers["filename"].map(
        manifest.set_index("filename")["sequence_index"]
    ))
    headers = headers.sort_values("sequence_index").reset_index(drop=True)
display(headers)

## 7. Export processed data for Notebook 2

Outputs:

- `manifest.csv` — one row per source file, with reconstructed stage/order;
- `validation.csv` — all automated checks;
- `gamry_headers.csv` — original header metadata;
- `experiment_config.json` — experiment constants;
- `import_summary.json` — provenance and export inventory;
- `tables/*.csv` — one clean table for every table in every `.DTA` file;
- `table_index.csv` — mapping from each exported table to its raw source.

CSV files are used instead of a Python-only binary format so every result remains auditable.

In [ ]:
def safe_slug(text):
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._")
    return slug or "table"


tables_dir = OUTPUT_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
table_index_rows = []

for filename, parsed in parsed_files.items():
    source_row = (
        manifest.loc[manifest["filename"].eq(filename)].iloc[0].to_dict()
        if not manifest.empty else {}
    )
    for table_number, table in enumerate(parsed["tables"], start=1):
        export_name = (
            f"{int(source_row.get('sequence_index', 0)):03d}__"
            f"{safe_slug(Path(filename).stem)}__"
            f"{table_number:02d}_{safe_slug(table['name'])}.csv"
        )
        export_path = tables_dir / export_name
        table["data"].to_csv(export_path, index=False)
        table_index_rows.append({
            "source_filename": filename,
            "source_stage": source_row.get("stage"),
            "sequence_index": source_row.get("sequence_index"),
            "table_number": table_number,
            "table_name": table["name"],
            "row_count": len(table["data"]),
            "column_count": len(table["data"].columns),
            "columns": " | ".join(map(str, table["data"].columns)),
            "units_json": json.dumps(table["units"], ensure_ascii=False),
            "exported_csv": str(Path("tables") / export_name),
        })

table_index = pd.DataFrame(table_index_rows)
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
validation.to_csv(OUTPUT_DIR / "validation.csv", index=False)
headers.to_csv(OUTPUT_DIR / "gamry_headers.csv", index=False)
table_index.to_csv(OUTPUT_DIR / "table_index.csv", index=False)

with (OUTPUT_DIR / "experiment_config.json").open("w", encoding="utf-8") as handle:
    json.dump(EXPERIMENT, handle, indent=2, ensure_ascii=False)

summary = {
    "schema_version": "1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_scope": "Gamry import and validation only",
    "raw_data_directory": str(RAW_DATA_DIR.resolve()),
    "raw_file_count": len(dta_files),
    "exported_table_count": len(table_index),
    "validation_status_counts": (
        validation["status"].value_counts().to_dict() if not validation.empty else {}
    ),
    "python_version": sys.version,
    "pandas_version": pd.__version__,
}
with (OUTPUT_DIR / "import_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, ensure_ascii=False)

print(f"Export complete: {OUTPUT_DIR.resolve()}")
display(table_index)

## 8. Handoff gate

Notebook 1 is complete when:

- all expected fixed stages show `PASS`;
- every raw file has at least one numeric table;
- any `REVIEW` items have been checked against the Gamry sequence;
- step CP, EIS, and CV filenames contain enough current information to pair them correctly;
- the `processed_data` folder contains the manifest, validation results, headers, configuration,
  table index, and exported tables.

Keep the original `.DTA` folder unchanged. Notebook 2 should read only the exported data and
will handle the last complete CV cycle, CP averaging windows, EIS replicate averaging/HFR,
Hg/HgO-to-RHE conversion, polarization curves, iR correction, and report figures.